In [2]:
import mimetypes
import pathlib
import shutil
import tempfile
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()

In [3]:
NBS_DIR = pathlib.Path().resolve()
REPO_DIR = NBS_DIR.parent
DATA_DIR = REPO_DIR / "data"
INPUT_DIR = DATA_DIR / "inputs"
OUTPUT_DIR = DATA_DIR / "outputs"
READY_DIR = DATA_DIR / "ready"

In [4]:
def perform_clear_and_optimize_image(image_path, output_path, max_size=(1920, 1920)):
    """
    Removes all metadata from an image (e.g. EXIF data).
    Optimizes the image file size while preserving quality and transparency when needed.
    """
    # Convert to Path objects
    image_path = pathlib.Path(image_path)
    output_path = pathlib.Path(output_path)
    
    # Open and create clean copy
    original = Image.open(image_path)

    # Determine if image has transparency
    has_transparency = (
        original.mode in ('RGBA', 'LA') or 
        (original.mode == 'P' and 'transparency' in original.info)
    )
    
    # Auto-rotate based on EXIF
    original = ImageOps.exif_transpose(original)

    # Resize if larger than max_size while maintaining aspect ratio
    if original.size[0] > max_size[0] or original.size[1] > max_size[1]:
        original.thumbnail(max_size, Image.Resampling.LANCZOS)

    # Convert mode based on transparency
    if has_transparency:
        if original.mode != 'RGBA':
            original = original.convert('RGBA')
        best_format = 'PNG'
    else:
        if original.mode in ('RGBA', 'P', 'LA'):
            original = original.convert('RGB')
        best_format = 'JPEG'

    # Save with optimized settings
    save_kwargs = {}
    if best_format == 'JPEG':
        save_kwargs.update({
            'quality': 85,
            'optimize': True,
            'progressive': True
        })
        output_path = output_path.with_suffix('.jpg')
    elif best_format == 'PNG':
        save_kwargs.update({
            'optimize': True,
            'compress_level': 6
        })
        output_path = output_path.with_suffix('.png')
    print(f'Saving {output_path}')
    original.save(output_path, format=best_format, **save_kwargs)
    return output_path

In [5]:
def perform_is_image(path, require_can_open=True):
    try:
        guessed_type, encoding = mimetypes.guess_type(path)
    except: 
        guessed_type = ""
    guessed_img = "image" in guessed_type
    if not guessed_img:
        return False
    if guessed_img and require_can_open:
        try:
            img_ = Image.open(path)
            print(img_)
        except:
            return False
    return True

In [6]:
img_file_paths = []
for file_path in INPUT_DIR.glob("*"):
    # print(file_path, file_path.stem, file_path.suffix)
    is_image = perform_is_image(file_path)
    if not is_image:
        continue
    start_output_path = READY_DIR / file_path.name
    final_output_path = perform_clear_and_optimize_image(file_path, start_output_path)
    img_file_paths.append(final_output_path)
print(img_file_paths)
# print(file_path.name, guessed_type)

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=3088x2316 at 0x23675C438D0>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\1.jpg
<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=2212x2566 at 0x236755A3FD0>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\10.png
<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=3284x1950 at 0x236755A1D90>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\11.png
<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1990x2532 at 0x236755A1D90>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\12.png
<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=2786x2028 at 0x23675C24210>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\13.png
<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=2102x2648 at 0x236755A8E10>
Saving D:\Resume-Projects\sellaiart-microservice\data\ready\14.png
<PIL.MpoImagePlugin.MpoImageFile image mode=RGB size=4032x3024 at 0x236755A1D90>
Saving D:\Resume-Projects

In [7]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [8]:
zip_outpath = OUTPUT_DIR / "images-optimized.zip"
zip_outpath.exists()

False

In [9]:
with tempfile.TemporaryDirectory() as temp_dir:
    for path in img_file_paths:
        shutil.copy(path, temp_dir)
    shutil.make_archive(zip_outpath.with_suffix(''), 'zip', temp_dir)